# Kaggle LoRA source\nTimezone Asia/Saigon. Run All does setup/preflight and optional 2-step smoke when Qwen assets exist. Full train is disabled. Do not upload from the agent session.\n\nStatuses: kaggle_execution_verified=false until an actual Kaggle run; linux_install_verified=false until a Linux pip-check receipt exists; ull_training_run_completed=false for this delivery.\n

## 1. Scope / status

In [ ]:
from pathlib import Path
import json, sys

def resolve_source_root() -> Path:
    for key in ('__vsc_ipynb_file__',):
        raw = globals().get(key)
        if raw:
            nb_path = Path(str(raw))
            if nb_path.suffix == '.ipynb' and nb_path.is_file():
                return nb_path.resolve().parent.parent
    for start in (Path.cwd(), *Path.cwd().parents):
        if (start / 'notebooks' / '03_kaggle_train.ipynb').is_file():
            return start
        if start.name == 'notebooks' and (start / '03_kaggle_train.ipynb').is_file():
            return start.parent
    raise SystemExit('source_root_unresolved: extract the source ZIP so notebooks/03_kaggle_train.ipynb exists')

SOURCE_ROOT = resolve_source_root()
print('source_root', SOURCE_ROOT)
print('python', sys.executable)
print('kaggle_execution_verified', False)
print('linux_install_verified', False)
print('full_training_run_completed', False)
print('next: select roots in the following cell')


## 2. Config roots\nWrite configs/kaggle.resolved.json. private_test_root is evaluation-only and must stay unmounted for python -m src.training train.

In [ ]:
import json, sys
from pathlib import Path

def resolve_source_root() -> Path:
    for key in ('__vsc_ipynb_file__',):
        raw = globals().get(key)
        if raw:
            nb_path = Path(str(raw))
            if nb_path.suffix == '.ipynb' and nb_path.is_file():
                return nb_path.resolve().parent.parent
    for start in (Path.cwd(), *Path.cwd().parents):
        if (start / 'notebooks' / '03_kaggle_train.ipynb').is_file():
            return start
        if start.name == 'notebooks' and (start / '03_kaggle_train.ipynb').is_file():
            return start.parent
    raise SystemExit('source_root_unresolved: extract the source ZIP so notebooks/03_kaggle_train.ipynb exists')

source_root = resolve_source_root()
kaggle_input = Path('/kaggle/input')
candidates = list(kaggle_input.glob('*')) if kaggle_input.exists() else []
print('input candidates', [str(p) for p in candidates] or 'none; using local private roots')
print('source ZIP lives under source_root; private G1 partitions are a separate attach')
manifests = list(kaggle_input.glob('*/prepared-manifest.json')) if kaggle_input.exists() else []
if len(manifests) > 1:
    raise SystemExit('select_private_inputs_explicitly:' + ','.join(str(item.parent) for item in manifests))
private_root = manifests[0].parent if len(manifests) == 1 else (source_root / 'data' / 'training-inputs' / 'v1')
qwen_candidates = [item for item in candidates if (item / 'config.json').is_file() and item.resolve() != (source_root / 'vendor' / 'e5-small-v2').resolve()]
qwen_assets = str(qwen_candidates[0]) if len(qwen_candidates) == 1 else None
if len(qwen_candidates) > 1:
    raise SystemExit('select_qwen_assets_explicitly:' + ','.join(str(item) for item in qwen_candidates))
resolved = {
  'schema_version': 'cs221-kaggle-resolved-v1',
  'source_root': str(source_root),
  'inference_root': str(source_root / 'data' / 'inference'),
  'private_train_root': str(private_root / 'private-train'),
  'private_dev_root': str(private_root / 'private-dev'),
  'knowledge_root': str(source_root / 'data' / 'knowledge-preparation'),
  'qwen_assets': qwen_assets,
  'e5_assets': str(source_root / 'vendor' / 'e5-small-v2'),
  'output_root': '/kaggle/working' if Path('/kaggle/working').exists() else str(source_root / 'artifacts' / 'kaggle-delivery' / 'v1'),
  'platform': sys.platform,
}
out = source_root / 'configs' / 'kaggle.resolved.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(resolved, indent=2), encoding='utf-8')
print('wrote', out)
print('private_root', private_root)
print('qwen_assets', qwen_assets)
print('private_test_root omitted from train/preflight config')
print('Qwen pin 989aa7980e4cf806f80c7fef2b1adb7bc71aa306')
print('E5 pin ffb93f3bd4047442299a41ebb6fa998a38507c52 (pretrained, not trained)')


## 3. Setup assets / deps\nMissing assets fail with pin, file, and config key. No global user-site. Candidate extra Python: configs/kaggle-requirements.in.

In [ ]:
from pathlib import Path
import subprocess, sys

def resolve_source_root() -> Path:
    for key in ('__vsc_ipynb_file__',):
        raw = globals().get(key)
        if raw:
            nb_path = Path(str(raw))
            if nb_path.suffix == '.ipynb' and nb_path.is_file():
                return nb_path.resolve().parent.parent
    for start in (Path.cwd(), *Path.cwd().parents):
        if (start / 'notebooks' / '03_kaggle_train.ipynb').is_file():
            return start
        if start.name == 'notebooks' and (start / '03_kaggle_train.ipynb').is_file():
            return start.parent
    raise SystemExit('source_root_unresolved')

source_root = resolve_source_root()
req = source_root / 'configs' / 'kaggle-requirements.in'
print('candidate requirements', req)
print('next: attach Qwen assets to local_qwen.assets / qwen_assets')
print('do not pip -U globally; do not use user-site')
if not (source_root / 'configs' / 'kaggle.resolved.json').is_file():
    raise SystemExit('missing configs/kaggle.resolved.json')


## 4. Data / schema / hash preflight

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-m', 'src.training', 'preflight', '--config', 'configs/kaggle.resolved.json', '--data-only']
print('running', cmd)
code = subprocess.call(cmd)
if code != 0:
    raise SystemExit(code)
print('preflight_ok')


## 5. GPU / runtime receipt

In [ ]:
import subprocess, sys
from pathlib import Path
script = Path('scripts/kaggle_gpu_runner.py')
code = subprocess.call([sys.executable, str(script)])
if code != 0:
    raise SystemExit(code)
print('gpu_receipt_recorded')


## 6. Two-optimizer-step smoke (only when Qwen assets exist)\nThis cell exits 0 with skipped_missing_assets when qwen_assets is null. It never full-trains.

In [ ]:
import json, subprocess, sys
from pathlib import Path
cfg = json.loads(Path('configs/kaggle.resolved.json').read_text(encoding='utf-8'))
assets = cfg.get('qwen_assets')
if not assets or not Path(assets).is_dir():
    print('skipped_missing_assets')
    print('pin=989aa7980e4cf806f80c7fef2b1adb7bc71aa306 file=qwen assets config_key=qwen_assets')
    print('tiny CPU mechanics remain available via python -m src.training train --mode smoke without qwen_assets')
else:
    code = subprocess.call([sys.executable, '-m', 'src.training', 'train', '--config', 'configs/kaggle.resolved.json', '--mode', 'smoke', '--max-steps', '2', '--run-id', 'technical-smoke'])
    if code != 0:
        raise SystemExit(code)
    print('qwen_smoke_finished')


## 7. Full train (DISABLED by default)\nSet ENABLE_FULL_TRAIN=True yourself after delivery. The agent must not run this.

In [ ]:
import json, subprocess, sys
from pathlib import Path
ENABLE_FULL_TRAIN = False
if not ENABLE_FULL_TRAIN:
    print('full train disabled; full_training_run_completed=false')
else:
    cfg = json.loads(Path('configs/kaggle.resolved.json').read_text(encoding='utf-8'))
    if not cfg.get('qwen_assets') or not Path(cfg['qwen_assets']).is_dir():
        raise SystemExit('MISSING_INPUT: attach qwen_assets before ENABLE_FULL_TRAIN')
    code = subprocess.call([sys.executable, '-m', 'src.training', 'train', '--config', 'configs/kaggle.resolved.json', '--mode', 'full', '--run-id', 'research-user'])
    raise SystemExit(code)


## 8. Resume / export / reload

In [ ]:
print('resume: python -m src.training train --mode full --resume latest-complete')
print('export: python -m src.training export --checkpoint best-dev --run-dir <research-run> --output artifacts/adapter')
print('verify: python -m src.training verify-export --export-dir artifacts/adapter --example-json <example>')
print('smoke adapters are smoke_only=true and are not research adapters')


## 9. Optional corpus / judging / evaluation after freeze\nUser-run. Historical freezes/F1.json and F2.json must not be overwritten.

In [ ]:
print('corpus: python -m src.corpus.automated_review --help')
print('judge: python -m src.annotations.local_judge --help')
print('new F1/F2 go to an explicit output root, never freezes/F1.json')


## 10. User-triggered download\nClick to copy /kaggle/working artifacts. This notebook does not upload.

In [ ]:
from pathlib import Path
print('download outputs from', Path('artifacts/kaggle-delivery/v1').resolve())
print('no credentials should appear in this notebook output')
